# Logical Routing for RAG

An implementation of **logical routing** — directing an incoming query to the appropriate
downstream chain (or data source) based on a classification made by the LLM itself, rather than
by keyword matching or manual rules.

**Why it helps:** many real RAG systems need to handle questions about more than one domain
(e.g. Python docs, SQL docs, Go docs). Rather than building one retriever that tries to cover
everything, logical routing asks the LLM to classify which domain a question belongs to
(using **structured output**, so the classification is a validated, typed object rather than
free text), then dispatches to the chain built specifically for that domain.

**Pipeline:**
1. Define a small **structured output schema** (`RouteQuery`) constraining the LLM's answer to
   one of a fixed set of known data sources.
2. Ask the LLM to classify a question against that schema.
3. Map the classification to the corresponding downstream chain with a routing function.
4. Wire it all into a single runnable chain: classify → route → (placeholder chain) run.

**Stack:** LangChain, Pydantic (structured output schema), Ollama as the local LLM.

> This notebook stops at routing to a placeholder string per data source (`"chain for sql_docs"`,
> etc.) — in a full system, each branch would be a real chain wired to a retriever scoped to
> that data source.


## Setup

Load model names from environment variables (`OLLAMA_MODEL` for the chat model) via a `.env` file, so the notebook doesn't hardcode model choices.

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

OLLAMA_MODEL = os.getenv("OLLAMA_MODEL")
EMBED_MODEL = os.getenv("OLLAMA_EMBED_LITE")

## 1. Define the Router

`RouteQuery` is a Pydantic model that constrains the LLM's output to exactly one of three known data sources (`Literal[...]`). Binding this schema to the LLM via `with_structured_output` means the model *must* return a valid, typed `RouteQuery` object instead of open-ended text — the classification is directly usable in code with no parsing needed.

In [2]:
from typing import Literal

from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field
from langchain_ollama import ChatOllama

# Data model
class RouteQuery(BaseModel):
    """Route a user query to the most relevant datasource."""

    datasource: Literal["python_docs", "sql_docs", "golang_docs"] = Field(
        ...,
        description="Given a user question choose which datasource would be most relevant for answering their question",
    )

# LLM with function call 
llm = ChatOllama(model=OLLAMA_MODEL, temperature=0,num_ctx=1024)
structured_llm = llm.with_structured_output(RouteQuery)

# Prompt 
system = """You are an expert at routing a user question to the appropriate data source.

Based on the programming language the question is referring to, route it to the relevant data source."""

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system),
        ("human", "{question}"),
    ]
)

# Define router 
router = prompt | structured_llm

c:\Users\chett\AppData\Local\Programs\Python\Python314\Lib\site-packages\langchain_core\utils\pydantic.py:42: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1 import BaseModel as BaseModelV1


## 2. Try the Router

Ask a SQL-related question and confirm the router correctly classifies it as `sql_docs`.

In [3]:
question = """Why doesn't the following code work:

select cust_id, store_num, sum(sales) from table_a group by cust_id
"""

result = router.invoke({"question": question})
result


RouteQuery(datasource='sql_docs')

## 3. Dispatch on the Classification

`choose_route` maps the structured classification to the chain that should actually handle the question. Wrapping it in `RunnableLambda` lets it slot directly into a LangChain pipe (`|`), so the full chain reads as: classify the question, then route it.

In [4]:
def choose_route(result):
    if "python_docs" in result.datasource.lower():
        ### Logic here
        return "chain for python_docs"
    elif "sql_docs" in result.datasource.lower():
        ### Logic here
        return "chain for sql_docs"
    else:
        ### Logic here
        return "chain for golang_docs"

from langchain_core.runnables import RunnableLambda

full_chain = router | RunnableLambda(choose_route)


## 4. Run the Full Chain

End-to-end: classify the question, then dispatch to its matching chain.

In [5]:
full_chain.invoke({"question": question})

'chain for sql_docs'

## Next Steps

- Replace each placeholder string (`"chain for sql_docs"`, etc.) with a real chain wired to a retriever/vector store scoped to that data source.
- Add a fallback/`"unclear"` route for questions that don't clearly belong to any known data source, rather than always defaulting to the `else` branch.
- Combine logical routing with [semantic routing](#) (routing by embedding similarity to example queries) for domains where the language alone doesn't cleanly separate categories.